# 6. Topic modeling: PTM (Pseudo-document-based Topic Model) (Section 4.4, Tables 4-7)

tomotopy `PTModel` (seed=42, 100 iterations, one worker so results are deterministic). Coherence uses the top 10 words of each topic.

The number of topics k is swept over 2, 4, ..., 28 and the model with the highest C_V coherence is
selected. All metrics below are reported for that selected model.
See docs/REPRODUCIBILITY_NOTES.md, section 5, about the C_NPMI values in the paper.

In [ ]:
from utils import *
import tomotopy as tp
from gensim import corpora
from gensim.models import CoherenceModel

In [ ]:
summary = {}
for platform in ["whatsapp", "telegram"]:
    texts = tokenize_for_topics(load_dataset(platform)["text_processed"])
    dictionary = corpora.Dictionary(texts)

    sweep, fitted = [], {}
    for k in range(2, 30, 2):
        model = tp.PTModel(k=k, seed=SEED)
        for doc in texts:
            if doc:
                model.add_doc(doc)
        model.train(100, workers=1)
        topics = [[w for w, _ in model.get_topic_words(i, top_n=10)] for i in range(model.k)]
        coherence = {m: CoherenceModel(topics=topics, texts=texts, dictionary=dictionary,
                                       coherence=m).get_coherence() for m in ["c_v", "c_npmi"]}
        sweep.append({"k": k, **coherence}); fitted[k] = topics

    sweep = pd.DataFrame(sweep)
    best = sweep.loc[sweep["c_v"].idxmax()]
    print(sweep.round(4).to_string(index=False))
    print("\nselected k =", int(best["k"]))
    topics = fitted[int(best["k"])]
    summary[NAMES[platform]] = {"k": int(best["k"]), "c_npmi": best["c_npmi"], "c_v": best["c_v"],
                                "irbo": irbo(topics), "diversity": topic_diversity(topics)}

    out = results_folder("06_topics_ptm")
    sweep.to_csv(out / f"ptm_{NAMES[platform]}_k_sweep.csv", index=False)
    pd.DataFrame({"topic": range(len(topics)), "words": ["; ".join(t) for t in topics]}) \
      .to_csv(out / f"ptm_{NAMES[platform]}_topics.csv", index=False)

summary = pd.DataFrame(summary).round(3)
summary

## Comparison with the paper (Tables 4 and 5)

In [ ]:
compare_with_paper([
    ("FW.Br k", 2, summary.loc["k", "FW.Br"]), ("FW.Br C_V", 0.610, summary.loc["c_v", "FW.Br"]),
    ("FW.Br IRBO", 0.70, summary.loc["irbo", "FW.Br"]), ("FW.Br diversity", 0.95, summary.loc["diversity", "FW.Br"]),
    ("FW.Br C_NPMI", 0.061, summary.loc["c_npmi", "FW.Br"]),
    ("FT.Br k", 28, summary.loc["k", "FT.Br"]), ("FT.Br C_V", 0.563, summary.loc["c_v", "FT.Br"]),
    ("FT.Br IRBO", 0.94, summary.loc["irbo", "FT.Br"]), ("FT.Br diversity", 0.73, summary.loc["diversity", "FT.Br"]),
    ("FT.Br C_NPMI (paper value is from k=6, see notes)", 0.024, summary.loc["c_npmi", "FT.Br"]),
], tolerance=0.01)